In [1]:
import numpy as np
import os
import torch
import sys
sys.path.append("/data/nishome/user1/chaochuan/TSGym_benchmark")
from models.TSGym import Model as TSGym
print(torch.device("cuda" if torch.cuda.is_available() else "cpu"))

/data/nishome/user1/miniconda3/envs/mqenv/lib/python3.11/site-packages/local_attention/rotary.py:33: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled = False)
/data/nishome/user1/miniconda3/envs/mqenv/lib/python3.11/site-packages/local_attention/rotary.py:55: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled = False)


cuda


In [9]:
import sqlite3
import pandas as pd

dataset="ECL"
df = None
for dataset in ["ETTh1", "ETTh2", "ETTm1", "ETTm2", "ECL", "ECl", "Exchange", "ili", "nasdaq", "nyse", "weather", "traffic"]:
    DB_PATH = f"V100_log/long_term_forecast_SOTA_{dataset}_log.db"

    conn = sqlite3.connect(DB_PATH)
    temp_df = pd.read_sql_query("SELECT * FROM exp_logs", conn)
    conn.close()
    if df is None:
        df = temp_df
    else:
        df = pd.concat([df, temp_df], ignore_index=True)
df

,exp_setting,start_time,end_time,duration_sec,status,max_gpu_mem_MB,result_metric,error_msg
0,LTF_Autoformer_ETTh1_ftM_sl96_ll48_pl96_dm512_...,2026-01-06 10:32:19,None,NaN,RUNNING,NaN,None,None
1,LTF_Autoformer_ETTh1_ftM_sl96_ll48_pl192_dm512...,2026-01-06 10:32:34,None,NaN,RUNNING,NaN,None,None
2,LTF_Autoformer_ETTh1_ftM_sl96_ll48_pl336_dm512...,2026-01-06 10:32:41,None,NaN,RUNNING,NaN,None,None
3,LTF_Autoformer_ETTh1_ftM_sl96_ll48_pl720_dm512...,2026-01-06 10:32:48,2026-01-06 10:37:00,252.0,FINISHED,6512.0,"""file:ETTh1.csv, mse:0.5438016653060913, mae:0...",None
4,LTF_Crossformer_ETTh1_ftM_sl96_ll48_pl96_dm512...,2026-01-06 10:37:13,2026-01-06 10:38:20,67.0,FINISHED,1434.0,"""file:ETTh1.csv, mse:0.4095850884914398, mae:0...",None
...,...,...,...,...,...,...,...,...
861,LTF_DUET_traffic_ftM_sl96_ll48_pl192_dm512_el4...,2026-01-07 05:11:04,2026-01-07 05:11:18,14.0,FAILED,NaN,null,CUDA out of memory. Tried to allocate 4.34 GiB...
862,LTF_DUET_traffic_ftM_sl96_ll48_pl336_dm512_el4...,2026-01-07 05:11:24,2026-01-07 05:11:36,12.0,FAILED,NaN,null,CUDA out of memory. Tried to allocate 4.34 GiB...
863,LTF_DUET_traffic_ftM_sl96_ll48_pl720_dm512_el4...,2026-01-07 05:11:42,2026-01-07 05:11:50,8.0,FAILED,NaN,null,CUDA out of memory. Tried to allocate 4.34 GiB...
864,LTF_FiLM_traffic_ftM_sl720_ll48_pl96_dm512_el2...,2026-01-07 05:11:57,2026-01-07 11:38:22,23185.0,FINISHED,10544.0,"""file:traffic.csv, mse:0.4109334647655487, mae...",None


In [10]:
df[df['status']=="FINISHED"].shape, df[df['status']=="FAILED"].shape, df[df['status']=="RUNNING"].shape

((798, 8), (56, 8), (12, 8))

In [19]:
df[(df['status'].str.contains("FAILED", regex=True, na=False)) & ~(df['error_msg'].str.contains("mamba", regex=False, na=False)) & ~(df['error_msg'].str.contains("CUDA", regex=False, na=False))]

,exp_setting,start_time,end_time,duration_sec,status,max_gpu_mem_MB,result_metric,error_msg
644,LTF_Crossformer_nasdaq_ftM_sl36_ll18_pl48_dm51...,2026-01-06 10:55:00,2026-01-06 10:55:01,1.0,FAILED,NaN,null,__len__() should return >= 0
648,LTF_DLinear_nasdaq_ftM_sl36_ll18_pl48_dm512_el...,2026-01-06 10:55:51,2026-01-06 10:55:51,0.0,FAILED,NaN,null,__len__() should return >= 0
662,LTF_SegRNN_nasdaq_ftM_sl36_ll48_pl24_dm512_el2...,2026-01-06 10:59:43,2026-01-06 10:59:44,1.0,FAILED,NaN,null,"Expected hidden size (1, 160, 512), got [1, 24..."
663,LTF_SegRNN_nasdaq_ftM_sl36_ll48_pl36_dm512_el2...,2026-01-06 10:59:51,2026-01-06 10:59:51,0.0,FAILED,NaN,null,"Expected hidden size (1, 160, 512), got [1, 24..."
664,LTF_SegRNN_nasdaq_ftM_sl36_ll48_pl48_dm512_el2...,2026-01-06 10:59:58,2026-01-06 10:59:59,1.0,FAILED,NaN,null,"Expected hidden size (1, 320, 512), got [1, 48..."
665,LTF_SegRNN_nasdaq_ftM_sl36_ll48_pl60_dm512_el2...,2026-01-06 11:00:09,2026-01-06 11:00:09,0.0,FAILED,NaN,null,"stack expects each tensor to be equal size, bu..."
666,LTF_TimeMixer_nasdaq_ftM_sl36_ll0_pl24_dm16_el...,2026-01-06 11:00:17,2026-01-06 11:00:17,0.0,FAILED,NaN,null,The size of tensor a (4) must match the size o...
667,LTF_TimeMixer_nasdaq_ftM_sl36_ll0_pl36_dm16_el...,2026-01-06 11:00:24,2026-01-06 11:00:25,1.0,FAILED,NaN,null,The size of tensor a (4) must match the size o...
668,LTF_TimeMixer_nasdaq_ftM_sl36_ll0_pl48_dm16_el...,2026-01-06 11:00:32,2026-01-06 11:00:32,0.0,FAILED,NaN,null,The size of tensor a (4) must match the size o...
669,LTF_TimeMixer_nasdaq_ftM_sl36_ll0_pl60_dm16_el...,2026-01-06 11:00:39,2026-01-06 11:00:40,1.0,FAILED,NaN,null,The size of tensor a (4) must match the size o...


In [18]:
df[(df['status'].str.contains("FAILED", regex=True)) & (df['error_msg'].str.contains("CUDA", regex=True))]

,exp_setting,start_time,end_time,duration_sec,status,max_gpu_mem_MB,result_metric,error_msg
426,LTF_Crossformer_ECL_ftM_sl96_ll48_pl720_dm256_...,2026-01-06 14:59:53,2026-01-06 14:59:57,4.0,FAILED,NaN,null,CUDA out of memory. Tried to allocate 302.00 M...
475,LTF_DUET_ECl_ftM_sl512_ll48_pl96_dm512_el4_dl1...,2026-01-06 15:04:09,2026-01-06 18:06:50,10961.0,FAILED,NaN,null,Attempting to deserialize object on CUDA devic...
858,LTF_Crossformer_traffic_ftM_sl96_ll96_pl720_dm...,2026-01-07 05:09:57,2026-01-07 05:10:04,7.0,FAILED,NaN,null,CUDA out of memory. Tried to allocate 1.58 GiB...
859,LTF_DUET_traffic_ftM_sl336_ll48_pl96_dm512_el4...,2026-01-07 05:10:11,2026-01-07 05:10:19,8.0,FAILED,NaN,null,CUDA out of memory. Tried to allocate 7.49 GiB...
860,LTF_DUET_traffic_ftM_sl96_ll48_pl96_dm512_el4_...,2026-01-07 05:10:26,2026-01-07 05:10:55,29.0,FAILED,NaN,null,CUDA out of memory. Tried to allocate 4.34 GiB...
861,LTF_DUET_traffic_ftM_sl96_ll48_pl192_dm512_el4...,2026-01-07 05:11:04,2026-01-07 05:11:18,14.0,FAILED,NaN,null,CUDA out of memory. Tried to allocate 4.34 GiB...
862,LTF_DUET_traffic_ftM_sl96_ll48_pl336_dm512_el4...,2026-01-07 05:11:24,2026-01-07 05:11:36,12.0,FAILED,NaN,null,CUDA out of memory. Tried to allocate 4.34 GiB...
863,LTF_DUET_traffic_ftM_sl96_ll48_pl720_dm512_el4...,2026-01-07 05:11:42,2026-01-07 05:11:50,8.0,FAILED,NaN,null,CUDA out of memory. Tried to allocate 4.34 GiB...


In [14]:
df[(df['exp_setting'].str.contains(f"True_{dataset}", regex=True))]

,exp_setting,start_time,end_time,duration_sec,status,max_gpu_mem_MB,result_metric,error_msg
8,LTF_TSGym1000324_True_False_RevIN_DFT_False_se...,2026-01-05 04:53:05,2026-01-05 04:53:09,4.0,FAILED,NaN,null,CUDA out of memory. Tried to allocate 3.15 GiB...
9,LTF_TSGym1000324_True_False_RevIN_DFT_False_se...,2026-01-05 04:53:17,2026-01-05 04:53:21,4.0,FAILED,NaN,null,CUDA out of memory. Tried to allocate 3.13 GiB...
10,LTF_TSGym1000324_True_False_RevIN_DFT_False_se...,2026-01-05 04:53:28,2026-01-05 04:53:33,5.0,FAILED,NaN,null,CUDA out of memory. Tried to allocate 3.11 GiB...
11,LTF_TSGym1000324_True_False_RevIN_DFT_False_se...,2026-01-05 04:53:40,2026-01-05 04:53:44,4.0,FAILED,NaN,null,CUDA out of memory. Tried to allocate 3.04 GiB...
33,LTF_TSGym1000341_True_False_RevIN_MA_True_seri...,2026-01-05 10:18:20,None,NaN,RUNNING,NaN,None,None
34,LTF_TSGym1000299_True_False_None_MA_True_serie...,2026-01-05 10:18:27,None,NaN,RUNNING,NaN,None,None
35,LTF_TSGym1000341_True_False_RevIN_MA_True_seri...,2026-01-05 10:21:06,None,NaN,RUNNING,NaN,None,None
37,LTF_TSGym1000299_True_False_None_MA_True_serie...,2026-01-05 10:22:01,None,NaN,RUNNING,NaN,None,None
38,LTF_TSGym1000341_True_False_RevIN_MA_True_seri...,2026-01-05 10:23:28,None,NaN,RUNNING,NaN,None,None
39,LTF_TSGym1000299_True_False_None_MA_True_serie...,2026-01-05 10:25:40,None,NaN,RUNNING,NaN,None,None


In [33]:
import sqlite3
import pandas as pd

dataset="Exchange"

DB_PATH = f"../long_term_forecast_SOTA_{dataset}_log.db"

conn = sqlite3.connect(DB_PATH)
df = pd.read_sql_query("SELECT * FROM exp_logs", conn)
conn.close()
df

,exp_setting,start_time,end_time,duration_sec,status,max_gpu_mem_MB,result_metric,error_msg
0,LTF_CrossCrossModel_Exchange_ftM_sl96_ll48_pl9...,2026-01-06 20:06:11,2026-01-06 20:06:39,28.0,FINISHED,490.0,"""file:exchange_rate.csv, mse:0.085512906312942...",None
1,LTF_CrossCrossModel_Exchange_ftM_sl96_ll48_pl1...,2026-01-06 20:06:48,2026-01-06 20:07:15,27.0,FINISHED,490.0,"""file:exchange_rate.csv, mse:0.179240271449089...",None
2,LTF_CrossCrossModel_Exchange_ftM_sl96_ll48_pl3...,2026-01-06 20:07:23,2026-01-06 20:07:32,9.0,FINISHED,576.0,"""file:exchange_rate.csv, mse:0.334074467420578...",None
3,LTF_CrossCrossModel_Exchange_ftM_sl96_ll48_pl7...,2026-01-06 20:07:41,2026-01-06 20:08:14,33.0,FINISHED,670.0,"""file:exchange_rate.csv, mse:0.866330444812774...",None


In [9]:
def get_log_df(db_path):
    import sqlite3
    import pandas as pd

    conn = sqlite3.connect(db_path)
    df = pd.read_sql_query("SELECT * FROM exp_logs", conn)
    conn.close()
    return df

In [32]:
import os
import warnings
warnings.filterwarnings("ignore")
DB_PATH_LIST = [x for x in os.listdir("../") if "SOTA" in x and x.endswith("_log.db")]
log_df_list = [get_log_df(os.path.join("../", db_path)) for db_path in DB_PATH_LIST]
full_log_df = pd.concat(log_df_list, ignore_index=True)
full_log_df_CCModel = full_log_df[full_log_df['exp_setting'].str.contains("CrossCrossModel", regex=True)]
full_log_df_CCModel['dataset'] = full_log_df_CCModel['exp_setting'].apply(lambda x: x.split("_")[2])
full_log_df_CCModel['pred_len'] = full_log_df_CCModel['exp_setting'].apply(lambda x: x.split("_")[6])
full_log_df_CCModel['mse'] = full_log_df_CCModel['result_metric'].apply(lambda x: round(float(x.split("mse:")[1].split(",")[0]), 3) if pd.notna(x) else None)
full_log_df_CCModel['mae'] = full_log_df_CCModel['result_metric'].apply(lambda x: round(float(x.split("mae:")[1].split(",")[0]), 3) if pd.notna(x) else None)
full_log_df_CCModel[['dataset', 'mse', 'mae']].groupby(['dataset']).mean()

,mse,mae
dataset,,
ECL,NaN,NaN
ETTh1,0.46325,0.45400
ETTh2,0.37525,0.40225
ETTm1,0.39775,0.40800
ETTm2,0.29700,0.34000
Exchange,0.36625,0.40725
ili,2.07875,0.90175
weather,0.17900,0.22200


In [26]:
sota_result = pd.read_excel("./SOTA_result.xlsx")

In [28]:
sota_result.fillna(method="ffill", inplace=True)

In [29]:
sota_result

,Unnamed: 0,Unnamed: 1,DUET,Unnamed: 3,TimeMixer,Unnamed: 5,TSMixer,Unnamed: 7,MICN,Unnamed: 9,...,LightTS,Unnamed: 45,Informer,Unnamed: 47,Transformer,Unnamed: 49,Reformer,Unnamed: 51,CrossCrossModel,Unnamed: 53
0,NaN,NaN,mse,mae,mse,mae,mse,mae,mse,mae,...,mse,mae,mse,mae,mse,mae,mse,mae,mse,mae
1,dataset,pred_len,mse,mae,mse,mae,mse,mae,mse,mae,...,mse,mae,mse,mae,mse,mae,mse,mae,mse,mae
2,ETTm1,96,0.2929,0.3426,0.324,0.365,0.4894,0.4727,0.3206,0.3728,...,0.3603,0.3948,0.7418,0.6484,0.5051,0.4892,0.8972,0.6693,mse,mae
3,ETTm1,192,0.403,0.4034,0.3694,0.387,0.4744,0.4802,0.3685,0.4058,...,0.404,0.4193,0.893,0.7039,0.7589,0.6487,0.9136,0.6835,mse,mae
4,ETTm1,336,0.4277,0.4219,0.3915,0.4057,0.5374,0.5284,0.423,0.4488,...,0.4455,0.4507,1.0328,0.7749,1.0449,0.7919,1.0329,0.7461,mse,mae
5,ETTm1,720,0.5056,0.4691,0.4528,0.439,0.6059,0.5682,0.4953,0.4895,...,0.5423,0.5146,1.2065,0.8176,1.0352,0.7825,1.15,0.7931,mse,mae
6,ETTm1,Avg,0.407,0.409,0.384,0.399,0.527,0.512,0.402,0.429,...,0.438,0.445,0.969,0.736,0.836,0.678,0.998,0.723,mse,mae
7,ETTm2,96,0.1687,0.2553,0.1743,0.258,0.2533,0.3654,0.1854,0.2821,...,0.2254,0.3201,0.4915,0.5598,0.4297,0.468,0.7668,0.656,mse,mae
8,ETTm2,192,0.252,0.3119,0.2398,0.3001,0.4695,0.5327,0.2694,0.344,...,0.3261,0.3921,0.6148,0.6163,0.841,0.683,1.5306,0.9301,mse,mae
9,ETTm2,336,0.3276,0.3604,0.2982,0.3422,0.9014,0.7605,0.4008,0.4343,...,0.4954,0.4919,1.5991,0.9748,1.3972,0.9106,2.1139,1.0871,mse,mae
